# Fig. S6F: Viral genera enriched in the FN fraction

Summarize FN-enriched viral genera and host ranges for `FigS6F_plot.R`.
Run `Fig5H_taxonkit.sh` first.


In [1]:
import pandas as pd
import os

In [2]:
# Define paths
basedir = "/path/to/project"
directory_out = "FigS6F/"

# Output directory
path_out = f"{basedir}/output/revise_final/{directory_out}"

# Input files
path_in = f"{basedir}/output/revise_final/Fig5/"
taxonomy_dir = f"{basedir}/output/revise_final/Fig5HI/"
# Viral metadata
meta_in = f"{path_in}virus_all_4474.meta.txt"
# ICTV virus properties
ictv_path = f"{basedir}/data/virus_properties_ICTV.tsv"
# Virus–Host DB
hostdb_path = f"{basedir}/../../db/virushostdb_260821/virushostdb.tsv"
# Virus taxonomy
virustaxid = f"{taxonomy_dir}virus_taxonomy.tsv"
# Host taxonomy
hosttaxid = f"{taxonomy_dir}host_taxonomy.tsv"

if not os.path.exists(path_out):
    os.makedirs(path_out)
print("saving files:", path_out)


saving files: /Users/coco/iLab/itolab_backup/backup-latest/Lab/projects/2507blastx/output/revise_final/FigS6F/


# Read metadata

In [3]:
# Read viral metadata
df = pd.read_table(meta_in, dtype={"Project_Id": "Int64"}, index_col=0).drop("query_seq", axis=1)
print("original size:", df.shape)
# Exclude BLASTn hits to non-viral sequences
df_blastn = df.loc[~(df["is_virus_blastn"]==0), :].copy()
print("metadata size after removed blastn other hits:", df_blastn.shape)
df_blastn.head(3)

original size: (14353, 79)
metadata size after removed blastn other hits: (14036, 79)


,Project_Id,Project_Acc,Project_Title,Project_Description,ID,Host_species,BioProject_ID,Infection,species_host,tax_id,...,qcovs_blastn,qcovhsp_blastn,staxids_lst_blastn,name_lst_blastn,group_lst_blastn,taxid_blastn,family_blastn,genus_blastn,species_blastn,is_virus_blastn
0,515305,PRJNA515305,Analysis of possible mechanisms behind functio...,Long-term survival and integration of neural p...,SRR8445393,Rattus_norvegicus,PRJNA515305,Positive,Rattus_norvegicus,10116,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,515305,PRJNA515305,Analysis of possible mechanisms behind functio...,Long-term survival and integration of neural p...,SRR8445393,Rattus_norvegicus,PRJNA515305,Positive,Rattus_norvegicus,10116,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,515305,PRJNA515305,Analysis of possible mechanisms behind functio...,Long-term survival and integration of neural p...,SRR8445393,Rattus_norvegicus,PRJNA515305,Positive,Rattus_norvegicus,10116,...,99.0,99.0,['2748378'],['unclassified Cressdnaviricota family;unclass...,['unclassified Viruses domain'],2748378.0,unclassified Cressdnaviricota family,unclassified Cressdnaviricota genus,Cressdnaviricota sp.,1.0


# Count infection events

In [4]:
# Aggregate infection events by sample and genus
df_event = df_blastn.copy()
df_event = df_event[["ID", "cm", "family", "genus"]].groupby(["ID", "genus"]).agg(
    {"family": "first", "cm": "first"})
print("df event size:", df_event.shape)
print("#NaN:", df_event.isna().sum().sum())
df_event

df event size: (3292, 2)
#NaN: 0


family  cm
ID          genus                                                  
ERR10662039 Avastrovirus                           Astroviridae  FP
ERR1298522  Gammacoronavirus                      Coronaviridae  TP
            Megrivirus                           Picornaviridae  TP
            Orthoreovirus                       Spinareoviridae  TP
            Sicinivirus                          Picornaviridae  TP
...                                                         ...  ..
SRR9901050  unclassified Paramyxoviridae genus  Paramyxoviridae  FN
SRR9901053  Orthorubulavirus                    Paramyxoviridae  FN
            Rubulavirus                         Paramyxoviridae  FN
            unclassified Paramyxoviridae genus  Paramyxoviridae  FN
SRR9901058  Orthorubulavirus                    Paramyxoviridae  FN

[3292 rows x 2 columns]

In [5]:
# Count infection events by genus and fraction
df_event_count = df_event.reset_index()[["ID", "genus", "cm"]].groupby(["genus", "cm"]).count().unstack("cm", fill_value=0)
df_event_count.columns = df_event_count.columns.get_level_values(1) 
print("size of df_event:", df_event_count.shape)
print("total:", df_event_count.sum().sum(), "\n")  
df_event_count["total"] = df_event_count.sum(axis=1)


size of df_event: (246, 4)
total: 3292 



In [6]:
# Filter genera by the minimum event count
threshold = 5
df_event_count_filtered = df_event_count[df_event_count["total"]>=threshold].copy()
print("size of df_event_count_filtered:", df_event_count_filtered.shape)
print("total:", df_event_count_filtered["total"].sum())



size of df_event_count_filtered: (100, 5)
total: 3043


In [7]:
# Calculate infection-event proportions
df_event_prop = df_event_count_filtered[["TP", "FN", "TN", "FP"]].div(df_event_count_filtered[["TP", "FN", "TN", "FP"]].sum(axis=1), axis=0) * 100
df_event_prop = df_event_prop.sort_values("FN", ascending=False)
print("size of df_event_prop", df_event_prop.shape)


size of df_event_prop (100, 4)


# Add ICTV host ranges

In [8]:
# Classify ICTV host ranges
valid_host_groups = {
    "bacteria", "archaea", "fungi", "plants",
    "protists", "invertebrates", "vertebrates",
}


def classify_vertebrate(host):
    if pd.isna(host):
        return pd.NA

    groups = {
        x.strip().removeprefix("predicted ")
        for x in host.lower().split(",")
    }

    if not groups & valid_host_groups:
        return pd.NA

    return "vertebrates" in groups


# Map each genus to its family
genus_family = (
    df_event.reset_index()
    .loc[lambda x: x["genus"].isin(df_event_prop.index),
         ["genus", "family"]]
    .drop_duplicates()
)

if genus_family["genus"].duplicated().any():
    raise ValueError("Multiple families were found for the same genus.")

# Read ICTV family-level host information
ictv_host = (
    pd.read_table(ictv_path, index_col=0)
    .rename(columns={"Family": "family", "Host": "ictv_host"})
    [["family", "ictv_host"]]
)

# Add ICTV host ranges to the proportion table
host_range_table = (
    df_event_prop.rename_axis("genus")
    .reset_index()
    .merge(genus_family, on="genus", how="left", validate="one_to_one")
    .merge(ictv_host, on="family", how="left", validate="many_to_one")
)

host_range_table["is_vertebrate"] = (
    host_range_table["ictv_host"]
    .apply(classify_vertebrate)
    .astype("boolean")
)

# Add the total count for each genus
host_range_table["total_count"] = (
    host_range_table["genus"]
    .map(df_event_count_filtered["total"])
    .astype("Int64")
)

host_range_table = host_range_table[
    [
        "genus",
        "family",
        "ictv_host",
        "is_vertebrate",
        "total_count",
        *df_event_prop.columns,
    ]
]
# Sort by the FN proportion
host_range_table = host_range_table.sort_values("FN", ascending=False)



# Add Virus–Host DB host ranges

In [9]:
# Read Virus–Host DB
hostdb = pd.read_table(hostdb_path)

print("shape of hostdb:", hostdb.shape)
print(
    "NaN in tax id:",
    hostdb["virus tax id"].isna().sum(),
    hostdb["host tax id"].isna().sum()
)

virus_tax = pd.read_table(virustaxid)
host_tax = pd.read_table(hosttaxid)

# Add virus taxonomy
hostdb = hostdb.merge(
    virus_tax,
    left_on="virus tax id",
    right_on="virus_taxid",
    how="left"
)

# Add host taxonomy
hostdb = hostdb.merge(
    host_tax,
    left_on="host tax id",
    right_on="host_taxid",
    how="left"
)

print("shape after merge:", hostdb.shape)


# Rename the genus column
hostdb.columns = hostdb.columns.map(lambda x: "genus" if x == "virus_genus" else x)

# Join unique host classes for each viral genus
hostdb_genus = (
    hostdb[["genus", "host_class"]]
    .dropna(subset=["genus", "host_class"])
    .drop_duplicates()
    .sort_values(["genus", "host_class"])
    .groupby("genus", as_index=False)
    .agg(host_class=("host_class", ", ".join))
)

# Treat unclassified genera as having unknown hosts
is_unclassified_genus = (
    hostdb_genus["genus"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.startswith("unclassified", na=False)
)

hostdb_genus.loc[is_unclassified_genus, "host_class"] = pd.NA

print("Number of unique virus genera:", len(hostdb_genus))



shape of hostdb: (42704, 14)
NaN in tax id: 0 72
shape after merge: (42704, 30)
Number of unique virus genera: 3292


# Save table

In [10]:
# Add host information for each genus
host_range_table_class = host_range_table.merge(
    hostdb_genus,
    on="genus",
    how="left",
)


# Classify genera by whether known hosts include birds or mammals
def classify_aves_mammalia(host_class):
    # No known hosts
    if pd.isna(host_class):
        return "Unknown"

    # Split comma-separated host classes
    host_classes = [name.strip() for name in host_class.split(",")]

    # At least one avian or mammalian host
    if "Aves" in host_classes or "Mammalia" in host_classes:
        return "Yes"

    # Known hosts that exclude birds and mammals
    for name in host_classes:
        if name.lower() in ["", "nan"]:
            continue
        if name.lower().startswith("unclassified"):
            continue
        return "No"

    # No classified hosts
    return "Unknown"

host_range_table_class["Aves_Mammalia"] = (
    host_range_table_class["host_class"].apply(classify_aves_mammalia)
)

# Calculate FN - TP and sort in descending order
host_range_table_class["FN_minus_TP"] = (
    host_range_table_class["FN"] - host_range_table_class["TP"]
)

host_range_table_class = (
    host_range_table_class
    .sort_values("FN_minus_TP", ascending=False)
    .reset_index(drop=True)
)

# Save table
host_range_table_class.to_csv(
    f"{path_out}/infection_event_prop_ictv_virus_host_db.tsv",
    sep="\t",
    index=False,
)

display(host_range_table_class)

,genus,family,ictv_host,is_vertebrate,total_count,TP,FN,TN,FP,host_class,Aves_Mammalia,FN_minus_TP
0,Kahraivirus,Blumeviridae,bacteria,False,10,0.000000,100.000000,0.0,0.000000,unclassified root class,Unknown,100.000000
1,Triatovirus,Dicistroviridae,invertebrates,False,7,0.000000,100.000000,0.0,0.000000,Insecta,No,100.000000
2,Chahsmivirus,Fiersviridae,bacteria,False,8,0.000000,100.000000,0.0,0.000000,"Magnoliopsida, unclassified root class",No,100.000000
3,Gemycircularvirus,Genomoviridae,"fungi, plants, invertebrates, vertebrates",True,6,0.000000,100.000000,0.0,0.000000,"Arachnida, Aves, Insecta, Leotiomycetes, Magno...",Yes,100.000000
4,Mischivirus,Picornaviridae,vertebrates,True,10,0.000000,90.000000,0.0,10.000000,"Mammalia, unclassified root class",Yes,90.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
95,Sapovirus,Caliciviridae,vertebrates,True,24,70.833333,20.833333,0.0,8.333333,"Mammalia, unclassified root class",Yes,-50.000000
96,Betaarterivirus,Arteriviridae,vertebrates,True,162,70.987654,15.432099,0.0,13.580247,Mammalia,Yes,-55.555556
97,Sogarnavirus,Marnaviridae,protists,False,5,80.000000,0.000000,20.0,0.000000,"Coscinodiscophyceae, unclassified root class",No,-80.000000
98,Pasivirus,Picornaviridae,vertebrates,True,11,81.818182,0.000000,0.0,18.181818,Mammalia,Yes,-81.818182
